# 01 — Source and data quality (Phase 2 data-viability spike)

Answers the gate questions from `PLAN.md` Phase 2 for the PEGELONLINE
source: station identity, cadence, gaps, duplicates, timestamp offsets
(including DST transitions), historical depth, station overlap, and source
latency. The conclusions and the proceed/stop decision live in
`docs/data_viability_report.md`.

**Fixture-first:** by default every cell reads the committed fixtures under
`data_fixtures/pegelonline/` — no internet required. The live re-fetch at the
bottom is opt-in via an environment variable. All logic lives in
`scripts/source_spike.py`; cells only call and display (CLAUDE.md rule 17).

> RiverCast is **educational**. Water level is relative to the local gauge
> zero — it is not river depth, and these forecasts must never inform
> real-world decisions.

In [ ]:
import json
import os
import sys
from pathlib import Path

from rivercast.envcheck import find_lab_root

LAB_ROOT = find_lab_root(Path.cwd())
sys.path.insert(0, str(LAB_ROOT / "scripts"))
import source_spike

FIXTURES = source_spike.fixture_dir(LAB_ROOT)
print(f"lab root : {LAB_ROOT}")
print(f"fixtures : {FIXTURES.relative_to(LAB_ROOT)}")

## Station identity

Configured names resolved against the live station list (fixture
`stations_rhein.json`). Stations are pinned by immutable UUID in
`configs/stations.yaml` and `configs/base.yaml`; river kilometers confirm the
corridor runs upstream → downstream toward the KAUB target.

In [ ]:
rhein_stations = source_spike.load_json(FIXTURES / "stations_rhein.json")
configured = {"MAINZ", "OESTRICH", "BINGEN", "KAUB"}
for station in rhein_stations:
    if station["shortname"] in configured:
        print(
            f"{station['shortname']:<9} km {station['km']:<7} "
            f"uuid={station['uuid']}  number={station['number']}"
        )

## Series quality — recent and historical windows

`analyze_fixtures` recomputes every statistic deterministically from the
committed fixtures: cadence, gaps above tolerance, duplicates and conflicts,
value ranges, and the UTC offsets present in the raw timestamps.

In [ ]:
report = source_spike.analyze_fixtures(LAB_ROOT)

for section in ("recent_series", "historical_series_2024"):
    print(f"--- {section} ---")
    for name, stats in report[section].items():
        print(
            f"{name:<9} rows={stats['rows']:<5} cadence={stats['cadence_minutes_mode']}min "
            f"gaps={stats['gaps_over_tolerance']} dup={stats['duplicate_timestamps']} "
            f"conflicts={stats['conflicting_duplicates']} "
            f"range=[{stats['value_min']}, {stats['value_max']}]cm "
            f"offsets={stats['utc_offsets_seen']}"
        )

## Daylight-saving transitions and historical depth

The two DST windows straddle the 2025 spring-forward and fall-back
transitions. Source timestamps carry explicit ISO-8601 offsets (`+01:00` /
`+02:00`), so conversion to the internal UTC axis is deterministic — the grid
stays continuous, with no duplicated or impossible local times. The earliest
window shows usable data at 2000-01-01, i.e. ~26 years of history.

In [ ]:
for label in ("dst_spring_2025", "dst_fall_2025", "earliest_2000"):
    stats = report[label]
    print(
        f"{label:<16} rows={stats['rows']:<4} first={stats['first_utc']} "
        f"last={stats['last_utc']} gaps={stats['gaps_over_tolerance']} "
        f"conflicts={stats['conflicting_duplicates']} offsets={stats['utc_offsets_seen']}"
    )

print("\nraw rows around the 2025 fall-back transition (KAUB):")
fall_rows = source_spike.load_json(FIXTURES / "historical" / "dst_fall_2025_KAUB.json")
for row in fall_rows[54:64]:
    print(f"  {row['timestamp']}  ->  {row['value']} cm")

## Station overlap and upstream signal

All four stations must cover the same windows (coverage fraction 1.0 =
complete 15-minute grid in the common window). The correlation check asks the
spike's modeling question: does an upstream station's *current* 6-hour change
know more about KAUB's *future* 6-hour change than KAUB's own recent change
(the persistence view) does?

In [ ]:
print(json.dumps(report["recent_overlap"], indent=2))
print(json.dumps(report["historical_overlap_2024"], indent=2))

print("\n6-hour lead correlations vs KAUB future delta:")
print(f"  KAUB own past delta (persistence view): {report['auto_correlation_6h']['corr_own_delta_vs_future']}")
for name, result in report["upstream_correlation_6h"].items():
    print(f"  {name:<9} upstream delta: {result['corr_upstream_delta_vs_future']}  (n={result['n']})")

## Optional: live re-fetch

Re-runs the spike against PEGELONLINE and **rewrites the fixtures** (internet
required, ~20 requests). Only for maintainers refreshing the fixture set:
set `RIVERCAST_SPIKE_LIVE=1` before starting the kernel.

In [ ]:
if os.environ.get("RIVERCAST_SPIKE_LIVE") == "1":
    summary = source_spike.run_live_spike(LAB_ROOT)
    print(json.dumps(summary["latency_minutes"], indent=2))
else:
    print("skipped (fixture mode) — set RIVERCAST_SPIKE_LIVE=1 to re-fetch")

## Conclusion

All Phase 2 gate questions are answered in `docs/data_viability_report.md`,
including the proceed decision, the chosen training bootstrap window, and the
known risks (unvalidated raw data, 31-day REST window, browser-oriented
historical endpoint). Next: `02_features_and_leakage.ipynb` (Phase 5) after
the source adapters (Phase 3) and canonicalization (Phase 4) exist.